# 02d m9_pbm Complete Physical-Feature Ablation

**Research question.** How much predictive performance is retained when the
nine-feature physical score is reduced to a smaller, interpretable subset?

This notebook evaluates all $2^9-1=511$ nonempty feature subsets. Every subset
uses equal, unit-sum weights, selects its own best candidate window on every
day, and is evaluated with Beta-plus-Alpha leave-one-substation-out threshold
selection. This is an exploratory model-development comparison, not an untouched
external test.

**Inputs:** the 02b candidate/day caches.  
**Outputs:** the complete 511-row ranking, 4,088 fold thresholds, four compact
tables, three figures, and a manifest.  
**Expected runtime:** approximately 5-20 minutes after 02b exists.

## 1. Imports, Paths, And Ablation Contract

Feature names in the tables retain both their number and physical meaning.
Short forms such as F1+F3+F4 are used only in dense figure labels. The candidate
score is divided by feature count, so every subset remains on a comparable
roughly unit scale; this scaling does not change candidate ranking or thresholded
predictions relative to an equal-weight sum.

In [ ]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display


def find_notebook_directory() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if candidate.name == "notebooks" and (candidate / "_m9_pbm_data.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article" / "notebooks"
        if (nested / "_m9_pbm_data.py").exists():
            return nested
    raise FileNotFoundError("Could not locate the journal notebook directory.")


NOTEBOOK_DIR = find_notebook_directory()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _m9_pbm_data import (  # noqa: E402
    load_experiment_config,
    manifest_payload,
    output_dirs,
    resolve_paths,
    write_csv,
    write_manifest,
    write_parquet,
)
from _m9_pbm_features import FEATURE_COLUMNS, maximum_subset_scores  # noqa: E402
from _m9_pbm_plotting import (  # noqa: E402
    plot_ablation_by_feature_count,
    plot_ablation_feature_evidence,
    plot_top_ablation_subsets,
)
from _m9_pbm_validation import (  # noqa: E402
    all_nonempty_feature_subsets,
    assert_heldout_absent,
    binary_metrics,
    metric_rows,
    select_threshold,
    subset_equal_weight_matrix,
)

STARTED_AT = time.time()
ARTICLE_ROOT = NOTEBOOK_DIR.parent
CONFIG = load_experiment_config(ARTICLE_ROOT)
PATHS = resolve_paths(ARTICLE_ROOT, CONFIG)
SLUG = "02d_m9_pbm_feature_ablation"
OUTPUT_DIRS = output_dirs(PATHS, SLUG)

DEFINITIONS = all_nonempty_feature_subsets(FEATURE_COLUMNS)
WEIGHT_MATRIX = subset_equal_weight_matrix(DEFINITIONS, FEATURE_COLUMNS)
SCORE_COLUMNS = [f"subset_{mask:03d}" for mask in DEFINITIONS["subset_mask"]]
DEFINITIONS["score_column"] = SCORE_COLUMNS
DEFINITIONS["feature_set_short"] = [
    "+".join(
        f"F{index + 1}"
        for index, feature in enumerate(FEATURE_COLUMNS)
        if row[f"includes_{feature}"]
    )
    for _, row in DEFINITIONS.iterrows()
]

assert len(DEFINITIONS) == 511
assert DEFINITIONS["subset_mask"].nunique() == 511
assert np.allclose(WEIGHT_MATRIX.sum(axis=1), 1.0)
display(DEFINITIONS[["subset_mask", "feature_count", "feature_set", "feature_set_short"]].head())

## 2. Self-Consistent Subset Scoring

For subset $A$ with $|A|$ active features,

$$
Score_A(W)=\frac{1}{|A|}\sum_{i\in A}F_i(W),
\qquad
W_{d,A}^*=\operatorname*{arg\,max}_{W}Score_A(W).
$$

**Notation**

| Symbol | Meaning |
|---|---|
| $A$ | One nonempty subset of the nine physical features. |
| $|A|$ | Number of active features in that subset. |
| $F_i(W)$ | Physical feature $i$ for candidate window $W$. |
| $Score_A(W)$ | Equal-weight mean physical score for candidate $W$. |
| $W_{d,A}^*$ | Candidate selected by subset $A$ on substation-day $d$. |

The matrix calculation below processes 32 days at a time. It never forms a
9.77-million by 511 matrix in memory. Only the maximum candidate score for each
day and subset is retained for threshold evaluation.

In [ ]:
CACHE_ROOT = PATHS.intermediate / "02b_m9_pbm_candidate_features"
PARTITION_DIR = CACHE_ROOT / "_partitions"
DAY_INPUT_CACHE = CACHE_ROOT / "day_input_cache.parquet"
SUBSET_DAILY_CACHE = OUTPUT_DIRS["intermediate"] / "all_subset_daily_scores.parquet"
assert PARTITION_DIR.exists() and DAY_INPUT_CACHE.exists(), "Run Notebook 02b first."

if SUBSET_DAILY_CACHE.exists() and CONFIG["execution"]["resume_validated_intermediates"]:
    daily_scores = pd.read_parquet(SUBSET_DAILY_CACHE)
else:
    score_parts = []
    candidate_paths = sorted(PARTITION_DIR.glob("*_candidates.parquet"))
    assert len(candidate_paths) == 18
    for candidate_path in candidate_paths:
        candidates = pd.read_parquet(
            candidate_path,
            columns=[
                "dataset",
                "substation_id",
                "date",
                "candidate_id",
                *FEATURE_COLUMNS,
            ],
        )
        keys, maxima = maximum_subset_scores(
            candidates,
            feature_columns=FEATURE_COLUMNS,
            weight_matrix=WEIGHT_MATRIX,
            batch_days=32,
        )
        score_parts.append(
            pd.concat(
                [keys, pd.DataFrame(maxima, columns=SCORE_COLUMNS)],
                axis=1,
            )
        )
        print(f"Scored {candidate_path.stem}", flush=True)
    daily_scores = pd.concat(score_parts, ignore_index=True)
    labels = pd.read_parquet(DAY_INPUT_CACHE)
    daily_scores = daily_scores.merge(
        labels[["dataset", "substation_id", "date", "true_day", "confidence"]],
        on=["dataset", "substation_id", "date"],
        how="left",
        validate="one_to_one",
    )
    write_parquet(daily_scores, SUBSET_DAILY_CACHE)

assert len(daily_scores) == sum(CONFIG["datasets"]["expected_substation_days"].values())
assert daily_scores[SCORE_COLUMNS].notna().all().all()
display(
    pd.Series(
        {
            "substation_days": len(daily_scores),
            "feature_subsets": len(SCORE_COLUMNS),
            "daily_score_values": len(daily_scores) * len(SCORE_COLUMNS),
        },
        name="ablation_cache",
    )
)

## 3. Beta-Plus-Alpha LOSO Threshold Selection

For each subset and outer held-out Beta substation, the threshold is selected
from all Alpha substations plus sure days from the other seven Beta substations.
Alpha and Beta receive equal total weight in the macro-substation objective.
The held-out Beta substation is predicted once, and its confidence is used only
afterward to create Beta sure and Beta all reports.

In [ ]:
base_columns = ["dataset", "substation_id", "date", "true_day", "confidence"]
alpha_mask = daily_scores["dataset"].eq("alpha")
beta_frame = daily_scores.loc[daily_scores["dataset"].eq("beta"), base_columns].copy()
beta_frame = beta_frame.reset_index(drop=True)
beta_substations = sorted(beta_frame["substation_id"].unique())
sure_mask = beta_frame["confidence"].eq("sure").to_numpy()

metric_rows_all = []
threshold_rows = []
for definition in DEFINITIONS.itertuples(index=False):
    score_column = definition.score_column
    alpha = daily_scores.loc[alpha_mask, base_columns].copy()
    alpha["score"] = daily_scores.loc[alpha_mask, score_column].to_numpy()
    beta_scores = daily_scores.loc[~alpha_mask, score_column].to_numpy(dtype=float)
    predictions = np.zeros(len(beta_frame), dtype=bool)

    for heldout_substation in beta_substations:
        heldout_mask = beta_frame["substation_id"].eq(heldout_substation).to_numpy()
        beta_training = beta_frame.loc[sure_mask & ~heldout_mask].copy()
        beta_training["score"] = beta_scores[sure_mask & ~heldout_mask]
        assert_heldout_absent(beta_training, heldout_substation)
        training = pd.concat([alpha, beta_training], ignore_index=True)
        selection = select_threshold(
            training,
            score_column="score",
            dataset_balanced=True,
        )
        predictions[heldout_mask] = beta_scores[heldout_mask] >= selection.threshold

        fold_sure = beta_frame.loc[heldout_mask & sure_mask].copy()
        fold_sure["predicted_day"] = predictions[heldout_mask & sure_mask]
        fold_metrics = binary_metrics(fold_sure["true_day"], fold_sure["predicted_day"])
        threshold_rows.append(
            {
                "subset_mask": definition.subset_mask,
                "feature_count": definition.feature_count,
                "feature_set": definition.feature_set,
                "feature_set_short": definition.feature_set_short,
                "heldout_substation": heldout_substation,
                "threshold": selection.threshold,
                "training_rows": len(training),
                "training_macro_f1": selection.metrics["macro_f1"],
                **{f"heldout_sure_{key}": value for key, value in fold_metrics.items()},
            }
        )

    evaluation = beta_frame.copy()
    evaluation["predicted_day"] = predictions
    sure_evaluation = evaluation.loc[evaluation["confidence"].eq("sure")]
    sure_rows = metric_rows(sure_evaluation)
    all_rows = metric_rows(evaluation)
    sure_pooled = sure_rows.loc[sure_rows["aggregation"].eq("pooled")].iloc[0]
    sure_macro = sure_rows.loc[sure_rows["aggregation"].eq("macro_substation")].iloc[0]
    all_pooled = all_rows.loc[all_rows["aggregation"].eq("pooled")].iloc[0]
    metric_rows_all.append(
        {
            "subset_mask": definition.subset_mask,
            "feature_count": definition.feature_count,
            "feature_set": definition.feature_set,
            "feature_set_short": definition.feature_set_short,
            **{f"beta_sure_{key}": sure_pooled[key] for key in [
                "support", "positive_support", "tp", "fp", "fn", "tn",
                "precision", "recall", "f1"
            ]},
            **{f"beta_sure_macro_{key}": sure_macro[key] for key in [
                "precision", "recall", "f1"
            ]},
            **{f"beta_all_{key}": all_pooled[key] for key in [
                "support", "positive_support", "tp", "fp", "fn", "tn",
                "precision", "recall", "f1"
            ]},
        }
    )
    if definition.subset_mask % 50 == 0:
        print(f"Evaluated {definition.subset_mask} of 511 subset masks", flush=True)

ablation_metrics = pd.DataFrame(metric_rows_all).sort_values(
    ["beta_sure_f1", "beta_sure_precision", "beta_sure_recall"],
    ascending=False,
    kind="mergesort",
).reset_index(drop=True)
thresholds = pd.DataFrame(threshold_rows)
assert len(ablation_metrics) == 511
assert len(thresholds) == 511 * 8
display(ablation_metrics.head(20))

## 4. Best Subsets And Feature Evidence

The best-by-size table answers how many features are needed. Top-subset
frequency asks which features recur among strong models. Paired marginal effects
compare every nonempty subset that excludes a feature with the otherwise
identical subset that includes it. These summaries distinguish a feature that
is broadly helpful from one that is especially useful in a particular
interaction.

In [ ]:
best_by_count = (
    ablation_metrics.sort_values(
        ["feature_count", "beta_sure_f1", "beta_sure_precision"],
        ascending=[True, False, False],
        kind="mergesort",
    )
    .drop_duplicates("feature_count", keep="first")
    .sort_values("feature_count")
)

metric_by_mask = ablation_metrics.set_index("subset_mask")["beta_sure_f1"]
feature_rows = []
top_10 = set(ablation_metrics.head(10)["subset_mask"])
top_25 = set(ablation_metrics.head(25)["subset_mask"])
top_50 = set(ablation_metrics.head(50)["subset_mask"])
for feature_number, feature in enumerate(FEATURE_COLUMNS, start=1):
    bit = 1 << (feature_number - 1)
    deltas = []
    for subset_mask in range(1, 512):
        if subset_mask & bit:
            continue
        deltas.append(metric_by_mask.loc[subset_mask | bit] - metric_by_mask.loc[subset_mask])
    feature_rows.append(
        {
            "feature_number": feature_number,
            "feature": feature,
            "feature_short": f"F{feature_number}",
            "top_10_frequency_pct": 100 * sum(mask & bit > 0 for mask in top_10) / 10,
            "top_25_frequency_pct": 100 * sum(mask & bit > 0 for mask in top_25) / 25,
            "top_50_frequency_pct": 100 * sum(mask & bit > 0 for mask in top_50) / 50,
            "paired_comparisons": len(deltas),
            "mean_paired_delta_f1": float(np.mean(deltas)),
            "median_paired_delta_f1": float(np.median(deltas)),
            "positive_paired_delta_pct": 100 * float(np.mean(np.asarray(deltas) > 0)),
        }
    )
feature_evidence = pd.DataFrame(feature_rows)
paired_effects = feature_evidence[
    [
        "feature_number", "feature", "paired_comparisons", "mean_paired_delta_f1",
        "median_paired_delta_f1", "positive_paired_delta_pct"
    ]
].copy()

compact_mask = (1 << 0) | (1 << 2) | (1 << 3)
compact_current = ablation_metrics.loc[ablation_metrics["subset_mask"].eq(compact_mask)].iloc[0]
legacy = CONFIG["m9_pbm"]["ablation"]["legacy_fixed_window_compact_anchor"]
best_current = ablation_metrics.iloc[0]
compact_comparison = pd.DataFrame(
    [
        {
            "comparison": "self_consistent_F1_F3_F4",
            "feature_set": compact_current["feature_set"],
            "precision": compact_current["beta_sure_precision"],
            "recall": compact_current["beta_sure_recall"],
            "f1": compact_current["beta_sure_f1"],
            "status": "primary_recomputed",
        },
        {
            "comparison": "best_self_consistent_subset",
            "feature_set": best_current["feature_set"],
            "precision": best_current["beta_sure_precision"],
            "recall": best_current["beta_sure_recall"],
            "f1": best_current["beta_sure_f1"],
            "status": "primary_recomputed",
        },
        {
            "comparison": "legacy_fixed_window_F1_F3_F4_anchor",
            "feature_set": compact_current["feature_set"],
            "precision": legacy["precision"],
            "recall": legacy["recall"],
            "f1": legacy["f1"],
            "status": legacy["source"],
        },
    ]
)
display(
    best_by_count[
        [
            "feature_count",
            "feature_set_short",
            "beta_sure_precision",
            "beta_sure_recall",
            "beta_sure_f1",
        ]
    ]
)
display(feature_evidence)
display(compact_comparison)

## 5. Write Complete Results And Figures

The full 511-row ranking is compact enough for Git. The wide day-by-subset score
cache is reproducible and remains local. The legacy compact row is explicitly
marked as a recorded fixed-window regression anchor; it is not mixed into the
new ranking.

In [ ]:
METRICS_PATH = OUTPUT_DIRS["metrics"] / "01_all_511_subset_metrics.csv"
THRESHOLDS_PATH = OUTPUT_DIRS["metrics"] / "02_thresholds_by_subset_and_fold.csv"
BEST_PATH = OUTPUT_DIRS["tables"] / "table01_best_by_feature_count.csv"
FREQUENCY_PATH = OUTPUT_DIRS["tables"] / "table02_feature_frequency.csv"
EFFECT_PATH = OUTPUT_DIRS["tables"] / "table03_paired_marginal_effects.csv"
COMPACT_PATH = OUTPUT_DIRS["tables"] / "table04_compact_model_comparison.csv"
write_csv(ablation_metrics, METRICS_PATH)
write_csv(thresholds, THRESHOLDS_PATH)
write_csv(best_by_count, BEST_PATH)
write_csv(feature_evidence, FREQUENCY_PATH)
write_csv(paired_effects, EFFECT_PATH)
write_csv(compact_comparison, COMPACT_PATH)

FIGURE_COUNT = OUTPUT_DIRS["figures"] / "fig01_f1_by_feature_count.png"
FIGURE_TOP = OUTPUT_DIRS["figures"] / "fig02_top_subset_performance.png"
FIGURE_FEATURE = OUTPUT_DIRS["figures"] / "fig03_feature_frequency_and_marginal_effect.png"
plot_ablation_by_feature_count(ablation_metrics, best_by_count, FIGURE_COUNT)
plot_top_ablation_subsets(ablation_metrics, FIGURE_TOP)
plot_ablation_feature_evidence(feature_evidence, FIGURE_FEATURE)
display(FIGURE_COUNT)
display(FIGURE_TOP)
display(FIGURE_FEATURE)

## 6. Interpretation And Limitations

This ablation is the complete equal-weight subset search for the newly
self-consistent candidate cache. The central interpretation should compare the
compact F1/F3/F4 result with the best larger subsets and retain the distinction
between broad marginal usefulness and interaction-specific value.

Because the feature family and compact subset were informed by earlier Beta
development, even outer-held-substation predictions are development evidence.
An independent dataset remains necessary for final external validation.

In [ ]:
MANIFEST_OUTPUTS = [
    METRICS_PATH, THRESHOLDS_PATH, BEST_PATH, FREQUENCY_PATH, EFFECT_PATH,
    COMPACT_PATH, FIGURE_COUNT, FIGURE_TOP, FIGURE_FEATURE,
]
manifest = manifest_payload(
    paths=PATHS,
    config=CONFIG,
    started_at=STARTED_AT,
    inputs=[
        PATHS.config,
        CACHE_ROOT / "candidate_feature_cache.parquet",
        DAY_INPUT_CACHE,
    ],
    outputs=MANIFEST_OUTPUTS,
    row_counts={
        "feature_subsets": len(ablation_metrics),
        "outer_folds": len(thresholds),
        "daily_subset_scores": len(daily_scores) * len(SCORE_COLUMNS),
    },
)
manifest["local_intermediates"] = [str(SUBSET_DAILY_CACHE.relative_to(PATHS.article))]
MANIFEST_PATH = write_manifest(PATHS, f"{SLUG}.json", manifest)
inventory = pd.DataFrame({"path": [*MANIFEST_OUTPUTS, SUBSET_DAILY_CACHE, MANIFEST_PATH]})
inventory["exists"] = inventory["path"].map(Path.exists)
inventory["bytes"] = inventory["path"].map(lambda path: path.stat().st_size)
display(inventory)
assert inventory["exists"].all() and inventory["bytes"].gt(0).all()

## Fast Figure-Only Rerender

Run this cell after the lightweight setup cell whenever only the publication
figures need to change. It reads persisted results, refreshes validated
figure-source caches, and does not repeat candidate generation, fitting, or
evaluation.

In [ ]:
from _cached_figure_rendering import render_notebook_figures

RENDER_ONLY = True
if RENDER_ONLY:
    RENDERED_FIGURES = render_notebook_figures(
        ARTICLE_ROOT,
        '02d_m9_pbm_feature_ablation',
        refresh_sources=True,
    )
    display(pd.Series([str(path) for path in RENDERED_FIGURES], name="rendered_figure"))